# Model Building

# Importing Libraries

In [208]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report,accuracy_score,recall_score,r2_score,precision_score,roc_auc_score,confusion_matrix,\
f1_score,cohen_kappa_score

import warnings 
warnings.filterwarnings("ignore")

## Splitting the dataset

In [209]:
PROJECT_ROOT = Path.cwd().parent

train_df = pd.read_csv(PROJECT_ROOT / "data" / "train_sample_2024.csv")
test_df  = pd.read_csv(PROJECT_ROOT / "data" / "test_sample_2025.csv")

In [210]:
print(train_df.shape)
print(test_df.shape)

(200001, 25)
(100001, 25)


## Data Preprocessing


In [211]:
le = LabelEncoder()

train_df['MKT_UNIQUE_CARRIER'] = le.fit_transform(train_df['MKT_UNIQUE_CARRIER'])
test_df['MKT_UNIQUE_CARRIER'] = le.transform(test_df['MKT_UNIQUE_CARRIER'])

In [212]:
train_features = ['MONTH', 'DAY_OF_WEEK', 'MKT_UNIQUE_CARRIER', 'ORIGIN_AIRPORT_ID',
       'DEST_AIRPORT_ID', 'CRS_DEP_TIME','TAXI_IN','DISTANCE',
       'DISTANCE_GROUP']

x_train = train_df[train_features]
y_train = train_df['OPERATIONAL_RISK']

In [213]:
test_features =['MONTH', 'DAY_OF_WEEK', 'MKT_UNIQUE_CARRIER', 'ORIGIN_AIRPORT_ID',
       'DEST_AIRPORT_ID', 'CRS_DEP_TIME','TAXI_IN','DISTANCE',
       'DISTANCE_GROUP']


x_test = test_df[test_features]
y_test=test_df['OPERATIONAL_RISK']

In [214]:
summary = pd.DataFrame(columns=['Name','Accuracy','Precision','Recall','F1_Score','ROC_AUC','Cohen_Kappa'])

def metrics(name, y_test, ypred_hard, ypred_soft):
    global summary

    print(f"\nClassification Report for {name}:\n")
    print(classification_report(y_test, ypred_hard))
    print('------------------------------------------------------------')

    acc = round(accuracy_score(y_test, ypred_hard), 2)
    pre = round(precision_score(y_test, ypred_hard), 2)
    rec = round(recall_score(y_test, ypred_hard), 2)
    f1  = round(f1_score(y_test, ypred_hard), 2)
    auc = round(roc_auc_score(y_test, ypred_soft), 2)
    reliability = round(cohen_kappa_score(y_test, ypred_hard), 2)

    result = pd.DataFrame({
        'Name':[name],
        'Accuracy':[acc],
        'Precision':[pre],
        'Recall':[rec],
        'F1_Score':[f1],
        'ROC_AUC':[auc],
        'Cohen_Kappa':[reliability]
    })

    summary = pd.concat([summary, result], ignore_index=True)
    return summary

## Baseline Model - Logistic Regression

In [215]:
model1 = LogisticRegression(random_state=42)
model1.fit(x_train, y_train)

ypred_soft_1 = model1.predict_proba(x_test)[:,1]
ypred_hard_1 = (ypred_soft_1 > 0.5).astype(int)

summary = metrics(
    "Baseline Logistic",
    y_test,
    ypred_hard_1,
    ypred_soft_1
)


Classification Report for Baseline Logistic:

              precision    recall  f1-score   support

           0       0.79      1.00      0.88     78229
           1       0.69      0.04      0.07     21772

    accuracy                           0.79    100001
   macro avg       0.74      0.52      0.47    100001
weighted avg       0.77      0.79      0.70    100001

------------------------------------------------------------


In [216]:
summary

,Name,Accuracy,Precision,Recall,F1_Score,ROC_AUC,Cohen_Kappa
0,Baseline Logistic,0.79,0.69,0.04,0.07,0.63,0.05


## Model 2 - Logistic Regression with class weights

In [217]:
model2 = LogisticRegression(class_weight='balanced', random_state=42)
model2.fit(x_train, y_train)

ypred_soft_2 = model2.predict_proba(x_test)[:,1]
ypred_hard_2 = (ypred_soft_2 > 0.5).astype(int)

summary = metrics(
    "Logistic class",
    y_test,
    ypred_hard_2,
    ypred_soft_2
)


Classification Report for Logistic class:

              precision    recall  f1-score   support

           0       0.85      0.59      0.69     78229
           1       0.29      0.62      0.40     21772

    accuracy                           0.59    100001
   macro avg       0.57      0.60      0.55    100001
weighted avg       0.73      0.59      0.63    100001

------------------------------------------------------------


In [218]:
summary

,Name,Accuracy,Precision,Recall,F1_Score,ROC_AUC,Cohen_Kappa
0,Baseline Logistic,0.79,0.69,0.04,0.07,0.63,0.05
1,Logistic class,0.59,0.29,0.62,0.4,0.64,0.15


In [219]:
model3 = DecisionTreeClassifier(class_weight='balanced',random_state=42,max_depth=100)
model3.fit(x_train, y_train)

ypred_soft_3 = model3.predict_proba(x_test)[:,1]
ypred_hard_3 = (ypred_soft_3 > 0.5).astype(int)

summary = metrics(
    "Decision Tree",
    y_test,
    ypred_hard_3,
    ypred_soft_3
)


Classification Report for Decision Tree:

              precision    recall  f1-score   support

           0       0.80      0.81      0.80     78229
           1       0.28      0.27      0.27     21772

    accuracy                           0.69    100001
   macro avg       0.54      0.54      0.54    100001
weighted avg       0.68      0.69      0.69    100001

------------------------------------------------------------


In [220]:
summary

,Name,Accuracy,Precision,Recall,F1_Score,ROC_AUC,Cohen_Kappa
0,Baseline Logistic,0.79,0.69,0.04,0.07,0.63,0.05
1,Logistic class,0.59,0.29,0.62,0.4,0.64,0.15
2,Decision Tree,0.69,0.28,0.27,0.27,0.54,0.07


In [221]:
model4 = RandomForestClassifier(class_weight='balanced',random_state=42,max_depth=100)
model4.fit(x_train, y_train)

ypred_soft_4 = model4.predict_proba(x_test)[:,1]
ypred_hard_4 = (ypred_soft_4 > 0.5).astype(int)

summary = metrics(
    "Random Forest",
    y_test,
    ypred_hard_4,
    ypred_soft_4
)


Classification Report for Random Forest:

              precision    recall  f1-score   support

           0       0.79      0.98      0.88     78229
           1       0.50      0.09      0.15     21772

    accuracy                           0.78    100001
   macro avg       0.65      0.53      0.51    100001
weighted avg       0.73      0.78      0.72    100001

------------------------------------------------------------


In [222]:
summary

,Name,Accuracy,Precision,Recall,F1_Score,ROC_AUC,Cohen_Kappa
0,Baseline Logistic,0.79,0.69,0.04,0.07,0.63,0.05
1,Logistic class,0.59,0.29,0.62,0.4,0.64,0.15
2,Decision Tree,0.69,0.28,0.27,0.27,0.54,0.07
3,Random Forest,0.78,0.5,0.09,0.15,0.64,0.09


In [232]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
model5 = XGBClassifier(
    n_estimators=10000,
    
    scale_pos_weight=scale_pos_weight

)

model5.fit(x_train, y_train)
ypred_soft_5 = model5.predict_proba(x_test)[:,1]
ypred_hard_5 = (ypred_soft_5 > 0.5).astype(int)
summary = metrics(
    "XGBoost",
    y_test,
    ypred_hard_xgb,
    ypred_soft_xgb
)


Classification Report for XGBoost:

              precision    recall  f1-score   support

           0       0.80      0.96      0.87     78229
           1       0.46      0.12      0.19     21772

    accuracy                           0.78    100001
   macro avg       0.63      0.54      0.53    100001
weighted avg       0.72      0.78      0.72    100001

------------------------------------------------------------
